# Inverse Dynamics:
# Static Push Force

#### 1. Load scene

In [1]:
import os
import sys
import numpy as np
import time

import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *

In [2]:
xml_path = '../asset/panda_scene_with_box.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

In [3]:
""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data) # calculate initial dynamics

In [4]:
viewer = MUJOCOGLVIEWER(model, data)
mujoco.mj_resetData(model, data)
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # mujoco.mj_step(model, data)
    mujoco.mj_kinematics(model, data)
    viewer.render()

viewer.close()
del(viewer)

2026-03-04 09:42:47.088 python[55150:3232527] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


### 2. Push with Cartesian Force
- Set end effector push force
- Convert to joint torque with jacobian

In [5]:
""" GET SITE POSITION """
site_names = get_site_names(model, data)
print(site_names) # 'eef_site' , id 2
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
eef_site_pos = data.site_xpos[eef_site_id]
print("eef_site_pos: ", eef_site_pos)

""" GET SITE ROTATION  """
eef_site_rmat = data.site_xmat[eef_site_id]
eef_site_rmat = eef_site_rmat.reshape(3, 3) 
eef_site_euler = rmat2euler(eef_site_rmat)
print("eef_site_euler: ", eef_site_euler)

['right_center', 'eef_site']
eef_site_pos:  [ 3.92404417e-01 -1.57544621e-18  4.58585349e-01]
eef_site_euler:  [ 3.14159265e+00 -9.36953212e-17  2.85890788e-01]


In [6]:
""" DESIRED CARTESIAN FORCE """
desired_force = np.array([0.0, 0.0, -10.0]) # desired force in x direction

""" DESIRED TORQUE: MULTIPLY JACOBIAN TRANSPOSE """
# 1. get positional jacobian
jac_p = np.zeros((3, model.nv)) # positional jacobian
mujoco.mj_jacSite(model, data, jac_p, None, eef_site_id)
print("jac_p: ", jac_p)
# 2. get torque from jacobian transpose
torque_push  = jac_p.T @ desired_force
print("torque_push: ", torque_push)

jac_p:  [[ 1.57544621e-18  1.25585349e-01 -7.03495226e-18  1.91283348e-01
   1.51173858e-18  1.06500000e-01  1.54074396e-33]
 [ 3.92404417e-01  8.71312838e-17  4.04576097e-01  2.59544030e-16
   6.02192543e-02  8.71476723e-17  0.00000000e+00]
 [ 0.00000000e+00 -3.92404417e-01 -2.94186841e-17  4.71502326e-01
   1.77007039e-19  8.80000000e-02  0.00000000e+00]]
torque_push:  [ 0.00000000e+00  3.92404417e+00  2.94186841e-16 -4.71502326e+00
 -1.77007039e-18 -8.80000000e-01  0.00000000e+00]


#### 3. Main Loop
- Calculate desired torque every loop
- Clip to get stable torque

In [7]:
""" MAIN LOOP """
viewer = MUJOCOGLVIEWER(model, data)
mujoco.mj_resetData(model, data)
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

force_desired_1 = np.array([0.0, 0.0, -10.0]) # desired force in x direction
force_desired_2 = np.array([0.0, 0.0, -30.0]) # desired force in x direction

# viewer setting: foce visualization, collision group vis deactivate
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
viewer.options[0].geomgroup[0] = 0

force_flag = False

while viewer.is_alive():
    glfw.poll_events()
    if glfw.get_key(viewer.windows[0], glfw.KEY_SPACE) == glfw.PRESS:
        force_flag = not force_flag
        time.sleep(0.1)
    if force_flag:
        force_desired = force_desired_2
    else:
        force_desired = force_desired_1

    # 1. get current site position
    eef_site_pos = data.site_xpos[eef_site_id]
    # 2. get jacobian 
    jac_p = np.zeros((3, model.nv)) # positional jacobian
    mujoco.mj_jacSite(model, data, jac_p, None, eef_site_id)
    # 3. compute torque command
    torque_command = jac_p.T @ force_desired
    # 4. bias force
    torque_command += data.qfrc_bias

    # 5. apply control and render  
    data.ctrl[:] = torque_command
    mujoco.mj_step(model, data)
    viewer.render()

viewer.close()
del(viewer)